# How do you know synthetic tabular data is any good?

A small, transparent **evaluation harness for synthetic tabular data**, applied to the public
[Airline Satisfaction](https://www.kaggle.com/datasets/arseniyshutko/binary-aviation-satisfaction-129k) survey data (CC0).

I fit four deliberately simple generators on 60k real rows and score each one with four families of metrics.
A real hold-out set serves as the reference for what "ideal" looks like on every metric.

| family | question | ideal |
| --- | --- | --- |
| fidelity | do columns (and pairs of columns) look real? | TVD → 0, association error → 0 |
| detectability | can a classifier tell synthetic from real? | AUC → 0.5 |
| utility | does a model trained on synthetic work on real data (TSTR)? | close to train-real-test-real |
| privacy | are synthetic rows copies of training rows? | close_rate ≈ 0.05, exact copies → 0 |

The point is not the generators. It is showing **which metric catches which failure**, and that every
single metric can be fooled on its own.


In [ ]:
import numpy as np, pandas as pd, glob
from scipy import stats


## Generators

In [ ]:
"""Simple tabular data generators, each fitted on a real training frame.

They are deliberately simple and transparent so the evaluation harness can show *what* each metric catches:
- IndependentMarginals preserves every column distribution but destroys all dependencies.
- GaussianCopula preserves marginals and (rank) correlations, but not higher-order structure.
- NoisyCopy resamples real rows and re-draws a fraction of cells from the marginals. It looks excellent on
  fidelity and utility and fails on privacy. This is roughly how many "synthetic" competition datasets behave.
"""


import numpy as np
import pandas as pd
from scipy import stats


class Generator:
    name = "base"

    def fit(self, real: pd.DataFrame) -> "Generator":
        self.columns = list(real.columns)
        self.real = real.reset_index(drop=True)
        return self

    def sample(self, n: int, seed: int = 0) -> pd.DataFrame:
        raise NotImplementedError


class IndependentMarginals(Generator):
    name = "independent"

    def sample(self, n: int, seed: int = 0) -> pd.DataFrame:
        rng = np.random.default_rng(seed)
        return pd.DataFrame({c: self.real[c].to_numpy()[rng.integers(0, len(self.real), n)] for c in self.columns})


class GaussianCopula(Generator):
    """Map each column to normal scores through its empirical CDF, fit a correlation matrix, sample, invert.

    Categorical columns are integer-coded by frequency first, which makes the copula treat them as ordinal.
    This is a known weakness that the dependency metrics should expose.
    """

    name = "gaussian_copula"

    def fit(self, real: pd.DataFrame) -> "GaussianCopula":
        super().fit(real)
        self.categories = {}
        codes = {}
        for c in self.columns:
            col = self.real[c]
            if not pd.api.types.is_numeric_dtype(col):
                order = col.value_counts().index
                self.categories[c] = order
                codes[c] = pd.Categorical(col, categories=order).codes.astype(float)
            else:
                codes[c] = col.astype(float).fillna(col.median()).to_numpy()
        self.encoded = pd.DataFrame(codes)
        n = len(self.encoded)
        z = np.column_stack([stats.norm.ppf((stats.rankdata(self.encoded[c]) - 0.5) / n) for c in self.columns])
        self.corr = np.corrcoef(z, rowvar=False)
        self.sorted = {c: np.sort(self.encoded[c].to_numpy()) for c in self.columns}
        return self

    def sample(self, n: int, seed: int = 0) -> pd.DataFrame:
        rng = np.random.default_rng(seed)
        z = rng.multivariate_normal(np.zeros(len(self.columns)), self.corr, size=n)
        u = stats.norm.cdf(z)
        out = {}
        for j, c in enumerate(self.columns):
            ref = self.sorted[c]
            vals = ref[np.clip((u[:, j] * len(ref)).astype(int), 0, len(ref) - 1)]
            if c in self.categories:
                out[c] = self.categories[c][vals.astype(int)]
            else:
                out[c] = vals.astype(self.real[c].dtype) if self.real[c].notna().all() else vals
        return pd.DataFrame(out)


class NoisyCopy(Generator):
    name = "noisy_copy"

    def __init__(self, noise: float = 0.15):
        self.noise = noise
        self.name = f"noisy_copy_{int(noise * 100)}"

    def sample(self, n: int, seed: int = 0) -> pd.DataFrame:
        rng = np.random.default_rng(seed)
        base = self.real.iloc[rng.integers(0, len(self.real), n)].reset_index(drop=True).copy()
        for c in self.columns:
            mask = rng.random(n) < self.noise
            base.loc[mask, c] = self.real[c].to_numpy()[rng.integers(0, len(self.real), mask.sum())]
        return base


class HoldoutReal(Generator):
    """Reference point: genuinely new real rows. Every metric should look 'ideal' here."""

    name = "holdout_real"

    def __init__(self, holdout: pd.DataFrame):
        self.holdout = holdout.reset_index(drop=True)

    def sample(self, n: int, seed: int = 0) -> pd.DataFrame:
        return self.holdout.sample(n=min(n, len(self.holdout)), random_state=seed).reset_index(drop=True)


## Metrics

In [ ]:
"""Four families of metrics for synthetic tabular data, each returning plain floats.

fidelity        does each column, and each pair of columns, look like the real data?
detectability   can a classifier tell synthetic rows from real ones? (0.5 = indistinguishable)
utility         does a model trained on synthetic data work on real data? (train-synthetic-test-real)
privacy         are synthetic rows suspiciously close to, or exact copies of, real training rows?
"""


import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_predict
from sklearn.neighbors import NearestNeighbors


def _is_cat(s: pd.Series) -> bool:
    return not pd.api.types.is_numeric_dtype(s)


def _binned(real: pd.Series, other: pd.Series, bins: int = 20) -> tuple[np.ndarray, np.ndarray]:
    if _is_cat(real) or real.nunique() <= bins:
        cats = pd.Index(real.dropna().unique()).union(pd.Index(other.dropna().unique()))
        p = real.value_counts(normalize=True).reindex(cats, fill_value=0).to_numpy()
        q = other.value_counts(normalize=True).reindex(cats, fill_value=0).to_numpy()
        return p, q
    edges = np.unique(np.nanquantile(real, np.linspace(0, 1, bins + 1)))
    p = np.histogram(real.dropna(), edges)[0] / real.notna().sum()
    q = np.histogram(other.dropna().clip(edges[0], edges[-1]), edges)[0] / max(other.notna().sum(), 1)
    return p, q


def marginal_tvd(real: pd.DataFrame, synth: pd.DataFrame) -> float:
    """Mean total-variation distance between per-column distributions (0 = identical)."""
    return float(np.mean([0.5 * np.abs(np.subtract(*_binned(real[c], synth[c]))).sum() for c in real.columns]))


def _codes(df: pd.DataFrame, ref: pd.DataFrame) -> pd.DataFrame:
    out = {}
    for c in ref.columns:
        if _is_cat(ref[c]):
            out[c] = pd.Categorical(df[c], categories=ref[c].value_counts().index).codes.astype(float)
        else:
            out[c] = df[c].astype(float).fillna(ref[c].median())
    return pd.DataFrame(out)


def association_error(real: pd.DataFrame, synth: pd.DataFrame) -> float:
    """Mean absolute difference between pairwise Spearman correlation matrices (0 = same dependency structure)."""
    a = _codes(real, real).corr(method="spearman").to_numpy()
    b = _codes(synth, real).corr(method="spearman").to_numpy()
    iu = np.triu_indices_from(a, k=1)
    return float(np.nanmean(np.abs(a[iu] - b[iu])))


def detectability_auc(real: pd.DataFrame, synth: pd.DataFrame, seed: int = 0) -> float:
    """Cross-validated AUC of a real-vs-synthetic classifier. 0.5 is ideal; 1.0 means trivially detectable."""
    n = min(len(real), len(synth))
    both = pd.concat([real.sample(n, random_state=seed), synth.sample(n, random_state=seed)], ignore_index=True)
    X = _codes(both, real)
    label = np.r_[np.zeros(n), np.ones(n)]
    cat_mask = [_is_cat(real[c]) for c in real.columns]
    clf = HistGradientBoostingClassifier(max_iter=200, categorical_features=cat_mask, random_state=seed)
    p = cross_val_predict(clf, X, label, cv=3, method="predict_proba")[:, 1]
    return float(roc_auc_score(label, p))


def tstr_auc(synth: pd.DataFrame, real_test: pd.DataFrame, target: str, seed: int = 0) -> float:
    """Train on synthetic, test on real: ROC AUC of a model fitted only on synthetic rows."""
    feats = [c for c in real_test.columns if c != target]
    ref = real_test[feats]
    cat_mask = [_is_cat(ref[c]) for c in feats]
    clf = HistGradientBoostingClassifier(max_iter=300, categorical_features=cat_mask, random_state=seed)
    clf.fit(_codes(synth[feats], ref), synth[target].astype(int))
    return float(roc_auc_score(real_test[target].astype(int), clf.predict_proba(_codes(ref, ref))[:, 1]))


def privacy(synth: pd.DataFrame, real_train: pd.DataFrame, real_holdout: pd.DataFrame, n: int = 5000, seed: int = 0) -> dict:
    """Distance to closest record (DCR) of synthetic rows vs. genuinely unseen real rows.

    dcr_ratio = median DCR(synthetic → train) / median DCR(holdout → train). A ratio well below 1 means the
    generator is reproducing training rows more closely than new real data would, i.e. memorisation.
    """
    ref = real_train
    scale = _codes(ref, ref).std().replace(0, 1)
    nn = NearestNeighbors(n_neighbors=1).fit(_codes(ref, ref) / scale)
    s = synth.sample(min(n, len(synth)), random_state=seed)
    h = real_holdout.sample(min(n, len(real_holdout)), random_state=seed)
    d_s = nn.kneighbors(_codes(s, ref) / scale)[0][:, 0]
    d_h = nn.kneighbors(_codes(h, ref) / scale)[0][:, 0]
    # The median hides memorisation when only a minority of rows are copied, so also look at the tail:
    # how many synthetic rows sit closer to a training row than the closest 5% of genuinely new real rows?
    tail = np.quantile(d_h, 0.05)
    return {
        "dcr_median": float(np.median(d_s)),
        "dcr_ratio": float(np.median(d_s) / max(np.median(d_h), 1e-9)),
        "close_rate": float((d_s <= tail).mean()),  # 0.05 expected for a generator that does not memorise
        "exact_copy_rate": float((d_s < 1e-9).mean()),
    }


def evaluate(synth: pd.DataFrame, real_train: pd.DataFrame, real_test: pd.DataFrame, target: str) -> dict:
    return {
        "marginal_tvd": marginal_tvd(real_test, synth),
        "association_error": association_error(real_test, synth),
        "detectability_auc": detectability_auc(real_test, synth),
        "tstr_auc": tstr_auc(synth, real_test, target),
        **privacy(synth, real_train, real_test),
    }


## Run

In [ ]:
TARGET = "satisfaction"
real = pd.read_csv(glob.glob("/kaggle/input/**/data.csv", recursive=True)[0]).sample(frac=1, random_state=0).reset_index(drop=True)
real[TARGET] = real[TARGET].astype(int)
train, test, holdout = real.iloc[:60_000], real.iloc[60_000:90_000], real.iloc[90_000:]

rows = []
for g in [HoldoutReal(holdout), IndependentMarginals(), GaussianCopula(), NoisyCopy(0.15), NoisyCopy(0.5)]:
    synth = (g if isinstance(g, HoldoutReal) else g.fit(train)).sample(30_000, seed=0)
    rows.append({"generator": g.name, **evaluate(synth, train, test, TARGET)})
results = pd.DataFrame(rows).set_index("generator")
results.round(4)


## Reading the table

- **Independent marginals** gets near-perfect per-column fidelity (TVD ≈ 0.008, same as real data), yet a
  model trained on it is useless on real data (TSTR AUC ≈ 0.47) and a classifier spots it almost perfectly.
  *Per-column fidelity alone says nothing about usefulness.*
- **Gaussian copula** fixes most pairwise structure (association error drops roughly 5×) and recovers a lot of
  utility, but is still trivially detectable: rank correlations are not the whole joint distribution, and
  integer-coding categoricals forces a fake ordering on them.
- **Noisy copy (15%)** looks best on fidelity and utility (TSTR ≈ 0.987 vs 0.994 for real data) and is the
  hardest to detect. It is also a **privacy failure**: about 5% of its rows are exact copies of training rows,
  and about 15% sit closer to a training row than the closest 5% of genuinely new passengers.
  The *median* distance-to-closest-record ratio (≈ 1.2) does **not** reveal this. Only the tail metric does.
- **Noisy copy (50%)** trades that leak for much worse dependency structure and detectability.

**Takeaway for anyone building or judging synthetic data:** report all four families, include a real hold-out
reference row, and look at the tail of the privacy distribution, not just its median.


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
r = results.drop(index="holdout_real")
ref = results.loc["holdout_real"]
for ax, (col, title) in zip(axes, [("detectability_auc", "Detectability AUC (0.5 = ideal)"),
                                    ("tstr_auc", "Train-synthetic-test-real AUC"),
                                    ("close_rate", "Privacy: close_rate (0.05 = ideal)")]):
    r[col].plot.barh(ax=ax, color="#4c78a8")
    ax.axvline(ref[col], color="black", ls="--", lw=1, label="real hold-out")
    ax.set_title(title); ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), frameon=False)
plt.tight_layout(); plt.show()
